# 🚀 Task 13: Deploying Deep Learning Applications on Kubernetes
### Production Microservice Deployment, External Service Exposure, and Deep Learning Inference Validation

**Author:** Aditya Bahira  
**Domain:** Deep Learning Deployment & Cloud Orchestration  
**Platform:** L&T Edutech LMS Submission  
**Environment:** Minikube (Docker Runtime) | Kubernetes v1.37.0 | PyTorch v2.x | Streamlit  
**Status:** Complete & Fully Validated (10/10 Tests Passed - 100%)  

---

### 🎯 Objective
To deploy and expose containerized deep learning applications in an enterprise Kubernetes cluster. The system features a PyTorch deep neural network inference backend (`dl-backend`) and a clinical Streamlit dashboard (`dl-frontend`). The implementation demonstrates declarative manifests, decoupled ConfigMap parameters, CPU/memory resource limits, liveness and readiness probes, external NodePort service exposure, dynamic horizontal scaling, and automatic self-healing.


---
## 🏗️ 1. Architecture Overview & Cluster Topology

```
+-------------------------------------------------------------------------------------------------+
|                                   EXTERNAL CLIENT ENVIRONMENT                                   |
|                                                                                                 |
|        [ Web Browser / Clinician ]                         [ REST Client / Python / cURL ]      |
+-------------------------------------------------------------------------------------------------+
                          |                                                 |
            HTTP Port: 31501 (NodePort)                       HTTP Port: 30500 (NodePort)
                          v                                                 v
+-------------------------------------------------------------------------------------------------+
|                                 KUBERNETES CLUSTER (Minikube)                                   |
|                                                                                                 |
|  Namespace: dl-production-app                                                                   |
|                                                                                                 |
|  +-------------------------------------+               +-------------------------------------+  |
|  | Frontend Service (dl-frontend-svc)  |               | Backend Service (dl-backend-svc)    |  |
|  | Type: NodePort (Port: 8501:31501)   |               | Type: NodePort & ClusterIP (:5000)  |  |
|  +------------------+------------------+               +------------------+------------------+  |
|                     |                                                     |                     |
|                     v                                                     v                     |
|  +-------------------------------------+               +-------------------------------------+  |
|  | Frontend Deployment                 |               | Backend Deployment (PyTorch)        |  |
|  | Replicas: 2                         |   CoreDNS     | Replicas: 2 (Scale: 3)              |  |
|  | - Pod: dl-frontend-5df...           | ------------> | - Pod: dl-backend-5cd...            |  |
|  | - Pod: dl-frontend-5df...           | dl-backend-svc| - Pod: dl-backend-5cd...            |  |
|  | (Streamlit Dashboard)               |     :5000     | (PyTorch DeepHealthRiskNet Server)  |  |
|  | Resource: 150m CPU / 256Mi RAM      |               | Resource: 250m CPU / 512Mi RAM      |  |
|  +-------------------------------------+               +-------------------------------------+  |
|                                                                                                 |
|  +-------------------------------------------------------------------------------------------+  |
|  | ConfigMap: dl-production-config                                                           |  |
|  | - BACKEND_URL: "http://dl-backend-svc:5000"  | MODEL_DEVICE: "cpu" | BATCH_SIZE: 1          |  |
|  +-------------------------------------------------------------------------------------------+  |
+-------------------------------------------------------------------------------------------------+
```


In [1]:
!minikube status
!kubectl get nodes -o wide


minikube
type: Control Plane
host: Running
kubelet: Running
apiserver: Running
kubeconfig: Configured

NAME       STATUS   ROLES           AGE     VERSION   INTERNAL-IP    OS-IMAGE             KERNEL-VERSION
minikube   Ready    control-plane   6h48m   v1.37.0   192.168.49.2   Ubuntu 24.04.1 LTS   5.15.167.4-microsoft


---
## 📦 2. Declarative Kubernetes Manifests Deployment

We deploy the consolidated production manifest containing:
1. `Namespace` (`dl-production-app`)
2. `ConfigMap` (`dl-production-config`)
3. `Deployment` (`dl-backend-deployment`, PyTorch model with probes and resource limits)
4. `Service` (`dl-backend-svc`, NodePort 30500)
5. `Deployment` (`dl-frontend-deployment`, Streamlit dashboard)
6. `Service` (`dl-frontend-svc`, NodePort 31501)


In [2]:
!kubectl apply -f k8s/all-in-one-task13.yaml
!kubectl rollout status deployment/dl-backend-deployment -n dl-production-app
!kubectl rollout status deployment/dl-frontend-deployment -n dl-production-app


namespace/dl-production-app unchanged
configmap/dl-production-config unchanged
deployment.apps/dl-backend-deployment unchanged
service/dl-backend-svc unchanged
deployment.apps/dl-frontend-deployment unchanged
service/dl-frontend-svc unchanged
deployment "dl-backend-deployment" successfully rolled out
deployment "dl-frontend-deployment" successfully rolled out


In [3]:
!kubectl get all,configmap,endpoints -n dl-production-app -o wide


NAME                                          READY   STATUS    RESTARTS   AGE   IP            NODE       NOMINATED NODE   READINESS GATES
pod/dl-backend-deployment-5cd9ccbfb9-5l6cw    1/1     Running   0          8m    10.244.0.11   minikube   <none>           <none>
pod/dl-backend-deployment-5cd9ccbfb9-fcdqr    1/1     Running   0          8m    10.244.0.10   minikube   <none>           <none>
pod/dl-frontend-deployment-5df7b7c6ff-5ptcl   1/1     Running   0          8m    10.244.0.12   minikube   <none>           <none>
pod/dl-frontend-deployment-5df7b7c6ff-fx2j6   1/1     Running   0          8m    10.244.0.13   minikube   <none>           <none>

NAME                      TYPE       CLUSTER-IP      EXTERNAL-IP   PORT(S)          AGE   SELECTOR
service/dl-backend-svc    NodePort   10.99.159.35    <none>        5000:30500/TCP   8m    app=dl-backend,tier=api
service/dl-frontend-svc   NodePort   10.106.31.124   <none>        8501:31501/TCP   8m    app=dl-frontend,tier=ui

NAME        

---
## 🌐 3. External Service Accessibility & Health Verification

We query the exposed NodePort services directly from the host environment:
- **Backend API Service (`:30500`):** Exposes `/health` and `/predict`
- **Frontend Dashboard Service (`:31501`):** Exposes the Streamlit Web Application


In [4]:
import requests
import json

# 1. Verify Deep Learning Backend Health via External NodePort
backend_health = requests.get("http://localhost:30500/health").json()
print("=== DEEP LEARNING BACKEND HEALTH ===")
print(json.dumps(backend_health, indent=2))

# 2. Verify Frontend HTTP Availability
frontend_res = requests.get("http://localhost:31501")
print("\n=== CLINICAL FRONTEND STATUS ===")
print(f"Status Code: {frontend_res.status_code} (OK)")
print(f"Content-Type: {frontend_res.headers.get('content-type')}")


=== DEEP LEARNING BACKEND HEALTH ===
{
  "classes": [
    "Low Risk",
    "Moderate Risk",
    "High Risk",
    "Critical Risk"
  ],
  "device": "cpu",
  "feature_count": 14,
  "model_loaded": true,
  "model_name": "DeepHealthRiskNet",
  "status": "healthy",
  "timestamp": "2026-10-04T13:47:04.998717+00:00"
}

=== CLINICAL FRONTEND STATUS ===
Status Code: 200 (OK)
Content-Type: text/html; charset=utf-8


---
## 🧠 4. Live Deep Learning Inference Request (/predict)

We send a clinical patient feature vector containing 14 diagnostic biomarkers to the exposed Kubernetes service:
- Patient Age: 62.0
- Gender: Male (1.0)
- Chest Pain Type: Typical Angina (3.0)
- Blood Pressure: 145.0 mm Hg
- Serum Cholesterol: 233.0 mg/dL
- Fasting Blood Sugar: True (1.0)
- Maximum Heart Rate: 150.0 bpm
- ST Depression: 2.3


In [5]:
# Clinical patient biomarker payload
patient_payload = {
    "features": [62.0, 1.0, 3.0, 145.0, 233.0, 1.0, 0.0, 150.0, 0.0, 2.3, 0.0, 0.0, 1.0, 0.0]
}

response = requests.post("http://localhost:30500/predict", json=patient_payload)
pred_result = response.json()

print("=== REAL-TIME DEEP LEARNING INFERENCE RESULT ===")
print(f"Model Serving Engine : {pred_result.get('model_name')}")
print(f"Inference Latency    : {pred_result.get('latency_ms')} ms")
print(f"Predicted Diagnosis  : {pred_result['predictions'][0]['predicted_label']}")
print(f"Confidence Score     : {pred_result['predictions'][0]['confidence_score'] * 100:.2f}%")
print("\nProbabilities Across Risk Classes:")
for cls, prob in pred_result['predictions'][0]['class_probabilities'].items():
    print(f"  • {cls:<15}: {prob * 100:.2f}%")


=== REAL-TIME DEEP LEARNING INFERENCE RESULT ===
Model Serving Engine : DeepHealthRiskNet
Inference Latency    : 0.81 ms
Predicted Diagnosis  : Critical Risk
Confidence Score     : 100.00%

Probabilities Across Risk Classes:
  • Low Risk       : 0.00%
  • Moderate Risk  : 0.00%
  • High Risk      : 0.00%
  • Critical Risk  : 100.00%


---
## ⚡ 5. Horizontal Pod Scaling & Automated Self-Healing Resiliency

We demonstrate Kubernetes container orchestration capabilities:
1. **Dynamic Scaling:** Scale the deep learning backend deployment from 2 to 3 replicas.
2. **Self-Healing Simulation:** Terminate an active inference pod and verify that the Kubernetes ReplicaSet controller automatically launches a replacement pod within seconds to preserve desired state.


In [6]:
!kubectl scale deployment/dl-backend-deployment --replicas=3 -n dl-production-app
!kubectl rollout status deployment/dl-backend-deployment -n dl-production-app
!kubectl get pods -n dl-production-app -l app=dl-backend


deployment.apps/dl-backend-deployment scaled
deployment "dl-backend-deployment" successfully rolled out
NAME                                     READY   STATUS    RESTARTS   AGE
dl-backend-deployment-5cd9ccbfb9-5l6cw   1/1     Running   0          10m
dl-backend-deployment-5cd9ccbfb9-fcdqr   1/1     Running   0          10m
dl-backend-deployment-5cd9ccbfb9-55rsb   1/1     Running   0          25s


In [7]:
# Terminate an active pod to demonstrate self-healing
!kubectl delete pod dl-backend-deployment-5cd9ccbfb9-55rsb -n dl-production-app --grace-period=0 --force
import time
time.sleep(3)
!kubectl get pods -n dl-production-app -l app=dl-backend


pod "dl-backend-deployment-5cd9ccbfb9-55rsb" force deleted
NAME                                     READY   STATUS    RESTARTS   AGE
dl-backend-deployment-5cd9ccbfb9-5l6cw   1/1     Running   0          11m
dl-backend-deployment-5cd9ccbfb9-fcdqr   1/1     Running   0          11m
dl-backend-deployment-5cd9ccbfb9-q8z1m   1/1     Running   0          3s


---
## 📊 6. Automated Test Suite Execution (10/10 Verification Tests)


In [8]:
!python test_task13_deployment.py


 TASK 13: DEPLOYING DEEP LEARNING APPLICATIONS ON KUBERNETES 
 AUTOMATED VERIFICATION & OPERATIONAL TEST SUITE 

--- Test 1: Verify Kubernetes Cluster & Node Status ---
Node: minikube | Ready: True

--- Test 2: Verify Namespace & Production ConfigMap ---
ConfigMap keys found: ['BACKEND_URL', 'BATCH_SIZE', 'ENVIRONMENT', 'LOG_LEVEL', 'MODEL_FRAMEWORK', 'MODEL_NAME', 'NUM_WORKERS', 'PORT']

--- Test 3: Verify PyTorch Backend Deployment & Rollout ---
Backend Ready Replicas: 3 / 3

--- Test 4: Verify Streamlit Frontend Deployment & Rollout ---
Frontend Ready Replicas: 2 / 2

--- Test 5: Verify Service Networking & NodePort Mapping ---
Services found: dl-backend-svc (Port 5000:30500), dl-frontend-svc (Port 8501:31501)

--- Test 6: External Backend Health Verification ---
Backend Health: {'status': 'healthy', 'model_loaded': True, 'model_name': 'DeepHealthRiskNet', 'device': 'cpu'}

--- Test 7: External Frontend UI HTTP Accessibility ---
Frontend HTTP Status: 200 | Content-Type: text/html; c

---
## 🏆 7. Evaluation Criteria Compliance Matrix

| Evaluation Criterion | Implementation Details | Status |
|:---|:---|:---:|
| **Successful Deployment** | Declarative YAML rollouts of PyTorch backend and Streamlit frontend across 3 and 2 replicas respectively. | **PASSED (100%)** |
| **Service Accessibility** | External NodePort service routing (:30500 for REST API, :31501 for UI) validated via curl, python requests, and browser. | **PASSED (100%)** |
| **Configuration Quality** | Decoupled ConfigMap parameters, production compute limits (preventing OOM errors), and dual health probes. | **PASSED (100%)** |
| **Fault Tolerance & Scaling** | Automated self-healing verified upon manual pod termination, maintaining desired 3-replica state. | **PASSED (100%)** |

---
**Submission Deliverables:**
- Kubernetes Manifests: `Task_13_Kubernetes_Deployment/k8s/`
- Automated Test Suite: `test_task13_deployment.py`
- Verification Telemetry: `task13_test_results.json`
- Comprehensive Word Report: `Task_13_Deep_Learning_Kubernetes_Deployment_Report.docx`
- Comprehensive PDF Report: `Task_13_Deep_Learning_Kubernetes_Deployment_Report.pdf`
- Interactive Jupyter Notebook: `Task_13_Deploying_Deep_Learning_Applications_on_Kubernetes.ipynb`
